# 4. K-Means Clustering of Workout Types (Experiment)

**Purpose.** Fourth step of the RunSight training pipeline, and a documented negative result. It tests whether K-Means can automatically separate running workout types (easy run, tempo run, long run, interval) from summary features extracted from the sensor time series. Three attempts are made with different feature sets and cluster counts; none separates workout types reliably. Along the way it finds about 822 cycling workouts mislabeled as runs with simple rules. The conclusions point toward supervised or rule-based labeling instead.

**Inputs**
- `data/endomondoHR_speed.csv` - running workouts with the cleaned speed series (output of the earlier preprocessing notebooks)

**Outputs**
- None written to disk. Results are cluster summary tables and scatter plots shown in the notebook.

Part of RunSight - see the repo README.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

In [ ]:
PATH = "data/endomondoHR_speed.csv"

df = pd.read_csv(PATH)

# Clustering Workout Types with K-Means (Experiment)

The goal of this notebook is to automatically distinguish the running workouts in the Endomondo dataset (easy run, tempo run, long run, interval, etc.) with the **K-Means** clustering algorithm.

Below, in order:
1. Features will be extracted from the raw sensor data (time, speed, heart rate, location),
2. K-Means will be tried on these features,
3. The results will be interpreted, and we will discuss why it did not work / started to work,
4. Noise in the dataset (such as cycling workouts labeled as running) will be detected and cleaned.

This is not a linear "success story"; it is a **trial-and-error process** in which we form and test hypotheses step by step. At the end of each attempt you will see what did not work and how it steered the next attempt.

In [ ]:
num_cols = ['longitude', 'altitude', 'latitude', 'heart_rate', 'timestamp', 'speed']

In [ ]:
def parse_arr(s):
    if isinstance(s, np.ndarray):
        return s.astype(np.float64)
    cleaned = s.replace("[", "").replace("]", "").replace(",", " ")
    return np.fromstring(cleaned, sep=" ", dtype=np.float64)

In [ ]:
for col in num_cols:
    df[col] = df[col].apply(parse_arr)

## Step 1: Minimal Feature Extraction

We prepare the variables we will give to K-Means. The goal is first to run a simple experiment with the most basic running features and see whether clustering is possible at all.

The `extract_minimal_features` function returns a dictionary for each workout. It holds the following four values:

- **duration_min**: Duration of the workout in minutes
- **mean_speed**: Mean speed
- **mean_hr**: Mean heart rate
- **std_speed**: Standard deviation of speed (how variable is the speed during the workout?)

In [ ]:
def extract_minimal_features(row):
    ts = row['timestamp']
    spd = row['speed']
    hr = row['heart_rate']

    return {
        'duration_min': ((ts[-1] - ts[0])/60.0),
        'mean_speed': np.mean(spd),
        'mean_hr': np.mean(hr),
        'std_speed': np.std(spd)
    }

## Step 2: K-Means - Attempt 1

The minimal features obtained were assigned to the `X` DataFrame. Because K-Means is sensitive to outliers, the data was scaled with **RobustScaler** (which uses the median and the interquartile range instead of the mean and standard deviation, so a few extreme values do not distort the whole scaling).

**4 clusters** were considered appropriate to match the target running classes (easy run, tempo, long run, interval). For K-Means to work properly, it was given the scalar (numeric, summary) features extracted above rather than the raw time series.

### Interpreting the Output

The output showed us something important: although some features let us roughly distinguish groups dominated by a particular run type, the classes **cannot be separated clearly**.

- **duration_min (run duration)** was the feature that stood out at first glance and seemed to work partially for the long run class. But from a coach's point of view, not every 90-minute workout counts as a "long run". For example, a workout with a mean speed of 22 km/h and a mean heart rate of 136 may actually be a **cycling workout**, not a run.
- This points to another problem: filtering with `sport == 'run'` in the dataset brought us not only real runs but also mislabeled records (such as cycling).
- A 57-minute run with a mean heart rate of 149 bpm may point to a **tempo run**.
- Runs with high speed variance (std_speed) could have been the most important indicator of **interval workouts**. But there is a big trap here: since we have no age information, we cannot know each runner's heart rate/speed zones according to their own capacity. This can lead us into misclassification.

In [ ]:
print("Extracting 4 basic features...")
X = pd.DataFrame([extract_minimal_features(row) for _, row in df.iterrows()])

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

cluster_summary = X.copy()
cluster_summary['cluster'] = df['cluster']
print("\n--- Mean values of the 4 clusters ---")
print(cluster_summary.groupby('cluster').mean().round(2))

### Result: Attempt 1 Failed

As can be seen, this first clustering attempt failed. The classes could not be separated well enough; instead of separating clearly from each other, the clusters largely overlapped.

In [ ]:
xs = X_scaled[:,0]
ys = X_scaled[:,1]

plt.figure(figsize=(10, 6))
plt.scatter(xs, ys, c=df['cluster'], cmap='viridis')

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)

plt.show()

## Step 3: New Features and Dimensionality Reduction with PCA

To help K-Means cluster better, we add new features:

- **pca_route_magnitude**: `longitude`, `latitude` and `altitude` (the location data) were reduced to a single dimension with PCA, giving one number that summarizes how "large" or "spread out" the route is. The goal is, for example, to tell a long-distance route apart from a short route that stays roughly in place.
- **training_effect**: Added to make it easier to distinguish high-tempo (anaerobic) workouts from low-tempo (aerobic) ones. It represents the ratio of heart rate to speed (mean_hr / mean_speed).
- **duration**: Since each workout is recorded split into 500 pieces (data points), the total workout duration was computed by dividing the time difference by 500 so that this data could be used efficiently.

In [ ]:
def extract_features(row):
    ts = row['timestamp']
    spd = row['speed']
    hr = row['heart_rate']
    lon = row['longitude']
    lat = row['latitude']
    alt = row['altitude']

    dt = (ts[-1] - ts[0]) / 60


    spd = np.clip(spd, 6.0, 25.0)

    lat_rad = np.radians(lat[0])
    dx = (lon - lon[0]) * (np.cos(lat_rad) * 111139.0)
    dy = (lat - lat[0]) * 111139.0
    dz = alt - alt[0]

    coords_3d = np.column_stack((dx, dy, dz))
    pca = PCA(n_components=1)
    pca_result = pca.fit_transform(coords_3d)
    route_magnitude = float(np.std(pca_result))


    mean_speed = float(np.mean(spd))
    mean_hr = float(np.mean(hr))
    training_effect = (mean_hr / mean_speed) if mean_speed > 0.5 else 0.0

    return {
        'duration': dt,
        'mean_speed': np.mean(spd),
        'mean_hr': np.mean(hr),
        'std_speed': np.std(spd),
        'pca_route_magnitude': route_magnitude,
        'training_effect': training_effect
    }

## Step 4: K-Means - Attempt 2

This time a 3-class approach was tried. The main goal was to separate easy runs, long runs and anaerobic / high-tempo workouts. However, the outputs show that this approach is also insufficient.

A distinct "long run" group formed, but the speed variance (std_speed) was practically crushed here under the dominance of the `pca_route_magnitude` variable and lost its influence on the clustering - in other words, a single feature overshadowed the others.

In [ ]:
print("Extracting 3 basic features...")
X = pd.DataFrame([extract_features(row) for _, row in df.iterrows()])

scaler = RobustScaler(quantile_range=(5, 95))
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

cluster_summary = X.copy()
cluster_summary['cluster'] = df['cluster']
print("\n--- Mean values of the 3 clusters ---")
print(cluster_summary.groupby('cluster').mean().round(2))

### Result: Attempt 2 Was Also Insufficient

This approach actually led to an even worse clustering. The "simple is the best" principle still holds here, but K-Means alone is still far from solving our actual problem (classifying the workout type correctly).

In [ ]:
xs = X_scaled[:,0]
ys = X_scaled[:,1]

plt.figure(figsize=(10, 6))
plt.scatter(xs, ys, c=df['cluster'], cmap='viridis')

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)

plt.show()

## Step 5: Detecting Cycling Workouts

We realized that part of the problem in the previous attempts might come from noise caused by **cycling workouts** mislabeled in the dataset. In this step, we aim to detect, with simple rules, the records labeled as "running" that are actually cycling, and to clean them out.

In [ ]:
def is_likely_cycling(row):
    # Rule 1: speed is very high but HR is low (aerobic efficiency gap)
    if np.mean(row['speed']) > 22 and np.mean(row['heart_rate']) < 136:
        return True

    if np.max(row['speed']) > 30:
        return True

    return False

df['activity_type'] = df.apply(lambda x: 'cycling' if is_likely_cycling(x) else 'running', axis=1)

### Detection Result

Bad news: our suspicions turned out to be right. About **822 cycling workouts** are labeled as running in the dataset. This explains a significant part of the noise in the earlier clustering attempts.

In [ ]:
df['activity_type'].value_counts()

In [ ]:
df_runs = df[df['activity_type'] == 'running'].copy().reset_index(drop=True)

## Step 6: K-Means - Attempt 3 (with Cleaned Data)

As a last hope, K-Means is tried again on the `df_runs` dataset, from which the cycling workouts have been removed.

**Note:** The results of this attempt were not saved (i.e. no permanent change was made to `df_runs`), because this approach was seen to risk making the dataset imbalanced.

In [ ]:
print("Extracting 3 basic features...")
X = pd.DataFrame([extract_features(row) for _, row in df_runs.iterrows()])

scaler = RobustScaler(quantile_range=(5, 95))
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_runs['cluster'] = kmeans.fit_predict(X_scaled)

cluster_summary = X.copy()
cluster_summary['cluster'] = df_runs['cluster']
print("\n--- Mean values of the 3 clusters ---")
print(cluster_summary.groupby('cluster').mean().round(2))

### Result: Partial Progress, But Not Enough

Part of the noise at the top of the table does seem to have been cleaned. However, separating the data **fully** by workout type is still not possible.

The clustering succeeded along one particular direction, but this separation seems to follow not the **workout type** as we wanted, but only the **workout length**. This method is still very far from giving us a healthy, meaningful feature.

In [ ]:
xs = X_scaled[:,0]
ys = X_scaled[:,1]

plt.figure(figsize=(10, 6))
plt.scatter(xs, ys, c=df_runs['cluster'], cmap='viridis')

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)

plt.show()

## Overall Assessment

Throughout this notebook, K-Means was tried with three different feature sets, and none of them could reliably distinguish the workout type (easy run, tempo, long run, interval). The lessons we drew:

1. **Feature selection dominates clustering success.** A feature with a large scale difference (e.g. `pca_route_magnitude`) can overshadow other meaningful features (e.g. `std_speed`) even after scaling.
2. **Data quality comes before algorithm choice.** The `sport == 'run'` filter alone was not enough; about 822 cycling workouts were found labeled as running, and this noise was corrupting the results.
3. **The lack of contextual information (age, capacity, zone ranges) makes classification harder.** The same heart rate/speed values can point to different workout types for different runners.
4. **K-Means, as an unsupervised method, fell short here.** As a next step, labeled data or rule-based/supervised approaches with stronger domain knowledge should be tried.